# ATS Vibration Report — Priority Mismatch Detector (prototype)

Flags reports where the stated **Priority N** doesn't match what the **Recommendations** / **Comments** text implies.

**How this works (v1 scope):**
- Extracts text fields only (Recommendations, Comments, Equipment ID, Priority) — these are native PDF text, no OCR needed.
- Automatically detects and excludes "colored spectrum" style reports (per your instruction) using a simple image-colorfulness heuristic on the embedded chart screenshot. It does **not** read the spectrum/waterfall/trend charts themselves.
- Uses a **frozen pretrained sentence-embedding model** (not fine-tuned, not trained from scratch) to turn text into vectors, then trains a small classifier on top to predict "what priority does this text imply." This is the right choice at ~300-500 reports — a from-scratch deep model needs orders of magnitude more data.
- A report is flagged when the text-implied priority disagrees with the stated one, or when the model has low confidence that the text supports the stated priority.

**Known limitations to keep in mind:**
- The style-detection threshold is calibrated on only 1 example per style so far. Recalibrate with `scripts/calibrate_style_threshold.py` once you have more confirmed examples of each style.
- This prototype does not analyze the spectrum/waterfall/trend charts or amplitude type (velocity / acceleration / acceleration-enveloping) — text vs. stated priority only.
- Flag quality is only as good as your labeled validation set (Section 3/5) — that's what tells you if this is actually working, not just running.


In [ ]:
!pip install -q pymupdf pillow scikit-learn sentence-transformers joblib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Get the code + point at your data

Clone this repo (so `ats_priority_checker` is importable), and set `PDF_DIR` to wherever your report PDFs live in Drive. The repo is public, so no credentials are needed.


In [ ]:
import sys
from pathlib import Path

REPO_DIR = "/content/test-for-ats-ai-project"
REPO_URL = "https://github.com/r-chicken/Test-for-ATS-AI-Project.git"

if Path(REPO_DIR).exists():
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, REPO_DIR)


In [ ]:
from pathlib import Path

PDF_DIR = Path("/content/drive/MyDrive/ats_reports/pdfs")           # <-- your 300 (+200) report PDFs
OUT_DIR = Path("/content/drive/MyDrive/ats_reports/prototype_out")  # extracted CSVs + model saved here, persisted in Drive
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Extract text + filter report style from every PDF

This reads every PDF in `PDF_DIR`, parses out the fields, classifies each report's chart style, and writes three CSVs to `OUT_DIR`:
- `dataset.csv` — usable rows (waterfall-style, parsed cleanly)
- `excluded_style.csv` — colored-spectrum-style rows (excluded, per your instruction)
- `parse_errors.csv` — anything that didn't parse cleanly (review these — they usually mean the regex needs a small tweak for a report variant not seen yet)

`max_pages=1` below means only the first page of each PDF is read - some reports have a second page with nothing of note on it, so it's skipped entirely.


In [ ]:
from ats_priority_checker.dataset import build_dataset

summary = build_dataset(PDF_DIR, OUT_DIR, max_pages=1)
summary


In [ ]:
import pandas as pd

errors = pd.read_csv(OUT_DIR / "parse_errors.csv")
print(f"{len(errors)} rows need review")
errors[["source_file", "page_number", "parse_notes"]].head(20)


## 3. Build a hand-labeling sheet

You need at least a subset of reports where a human has judged whether the priority actually matches the writeup — that's the only way to know if the detector is working, and the only way to measure precision/recall. Start with ~100-150; you can label more later.

This exports a CSV with blank `human_label` (fill in `match`, `mismatch`, or `unsure`) and `human_notes` columns. Open it in Google Sheets, fill it in, then save it back (or update `LABELED_CSV` below to point at wherever you saved it).


In [ ]:
from ats_priority_checker.labeling import export_for_labeling

LABELING_CSV = OUT_DIR / "to_label.csv"
export_for_labeling(OUT_DIR / "dataset.csv", LABELING_CSV, sample_n=150)
print(f"Open and label: {LABELING_CSV}")


### After you've labeled the sheet

Set `LABELED_CSV` to wherever you saved your filled-in copy, then merge the labels back onto the full dataset.


In [ ]:
from ats_priority_checker.labeling import merge_labels

LABELED_CSV = OUT_DIR / "to_label.csv"  # update if you saved it elsewhere
labeled_df = merge_labels(OUT_DIR / "dataset.csv", LABELED_CSV, OUT_DIR / "dataset_with_labels.csv")


## 4. Train the text -> priority model

Note: this trains on the *stated* priority as the target for every report (that's free — every parsed report already has one). The human match/mismatch labels from Section 3 aren't used for training here; they're used in Section 5 purely to check whether "text disagrees with stated priority" actually lines up with human judgment.


In [ ]:
from ats_priority_checker.model import report_text, embed_texts, cross_validated_predictions

df = pd.read_csv(OUT_DIR / "dataset_with_labels.csv")
df = df.dropna(subset=["priority_num"]).reset_index(drop=True)

texts = df.apply(report_text, axis=1).tolist()
X = embed_texts(texts)
y = df["priority_num"].to_numpy()

cv_result = cross_validated_predictions(X, y, n_splits=5)
print(f"used {cv_result['n_splits']}-fold CV, classes seen: {cv_result['classes']}")


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y, cv_result["pred"]))


## 5. Flag mismatches, then validate against your hand labels

`flagged` has one row per report with `predicted_priority`, `confidence_in_stated_priority`, and `flag_mismatch`. Where you have a human label, compare directly — that precision/recall is the real signal for whether this prototype is usable, not the classification report above.


In [ ]:
from ats_priority_checker.model import flag_mismatches

flagged = flag_mismatches(df, cv_result["pred"], cv_result["proba"], cv_result["classes"])
flagged.to_csv(OUT_DIR / "flagged.csv", index=False)
flagged[["report_id", "priority_num", "predicted_priority", "confidence_in_stated_priority", "flag_mismatch", "flag_reason"]].head(20)


In [ ]:
labeled_subset = flagged.dropna(subset=["human_label"])
labeled_subset = labeled_subset[labeled_subset["human_label"] != ""]

if len(labeled_subset) == 0:
    print("No human labels yet - go label some reports in Section 3 to see real precision/recall here.")
else:
    from sklearn.metrics import precision_score, recall_score, confusion_matrix

    y_true = (labeled_subset["human_label"] == "mismatch").astype(int)
    y_pred = labeled_subset["flag_mismatch"].astype(int)
    print(f"n labeled = {len(labeled_subset)}")
    print(f"precision = {precision_score(y_true, y_pred, zero_division=0):.2f}")
    print(f"recall    = {recall_score(y_true, y_pred, zero_division=0):.2f}")
    print(confusion_matrix(y_true, y_pred))


## 6. Save the model (for scoring new reports later)

Fits on 100% of current data and saves to Drive. When you add the +200 reports (or more later), just rerun Sections 2 and 4-6 on the combined folder — same pipeline, bigger dataset.


In [ ]:
from ats_priority_checker.model import fit_final_model, save_bundle

final_clf = fit_final_model(X, y)
save_bundle(final_clf, OUT_DIR / "model" / "priority_classifier.joblib")
print("saved model to", OUT_DIR / "model" / "priority_classifier.joblib")


## 7. Scoring brand-new reports later

Once you have a saved model, score new PDFs without retraining:


In [ ]:
from ats_priority_checker.dataset import build_dataset
from ats_priority_checker.model import load_bundle, embed_texts, report_text

NEW_PDF_DIR = Path("/content/drive/MyDrive/ats_reports/new_pdfs")   # e.g. next month's batch
NEW_OUT_DIR = Path("/content/drive/MyDrive/ats_reports/new_out")

build_dataset(NEW_PDF_DIR, NEW_OUT_DIR)
new_df = pd.read_csv(NEW_OUT_DIR / "dataset.csv")

clf, embedding_model_name = load_bundle(OUT_DIR / "model" / "priority_classifier.joblib")
new_X = embed_texts(new_df.apply(report_text, axis=1).tolist(), model_name=embedding_model_name)

new_df["predicted_priority"] = clf.predict(new_X)
new_df["flag_mismatch"] = new_df["predicted_priority"] != new_df["priority_num"]
new_df.to_csv(NEW_OUT_DIR / "flagged.csv", index=False)
new_df[new_df["flag_mismatch"]][["report_id", "priority_num", "predicted_priority"]]
